# 02 — Models
Build the feature matrix and run walk-forward validation across all four models.

In [ ]:
import sys, warnings, time
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})


## Build feature matrix

In [ ]:
from src.data import load_data
from src.features import build_feature_matrix

factors, returns = load_data(use_synthetic=False)
X, y = build_feature_matrix(returns, factors, beta_window=36)

print("Feature matrix:", X.shape)
print("Features:", list(X.columns))
print("\nSample:")
X.head(3)


## Feature distributions
All features are cross-sectionally rank-normalised to [−0.5, +0.5] at each date.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(3, 3, figsize=(12, 8))
for ax, col in zip(axes.flatten(), X.columns):
    ax.hist(X[col].dropna(), bins=40, color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
fig.suptitle('Feature distributions (cross-sectionally rank-normalised)', fontsize=12)
plt.tight_layout()
plt.savefig('../results/figures/feature_dists.png', bbox_inches='tight')
plt.show()


## Walk-forward validation

Schema: 5-year train → 1-year validation (hyperparameter tuning) → 1-year test. Roll forward 1 year at a time.

> ⏱ This takes ~2–5 minutes depending on your machine.

In [ ]:
from src.models import OLS_FF5, LassoModel, RFModel, XGBModel, walk_forward

# Build raw returns panel for portfolio construction
fwd_raw   = returns.shift(-1)
raw_panel = fwd_raw.stack(future_stack=True).rename('raw_return')
raw_panel.index.names = ['Date', 'Asset']
raw_panel  = raw_panel.reindex(y.index).dropna()
common_idx = y.index.intersection(raw_panel.index)
y_aligned, X_aligned, raw_aligned = y.loc[common_idx], X.loc[common_idx], raw_panel.loc[common_idx]

models = [OLS_FF5(), LassoModel(), RFModel(), XGBModel()]

t0 = time.time()
results = walk_forward(X_aligned, y_aligned, models,
                       raw_returns=raw_aligned,
                       train_years=5, val_years=1, test_years=1)
print(f"\nCompleted in {time.time()-t0:.0f}s")


## Save results

In [ ]:
import pickle
with open('../results/walk_forward_results.pkl', 'wb') as f:
    pickle.dump({'results': results, 'X': X_aligned, 'y': y_aligned,
                 'raw_aligned': raw_aligned, 'feature_names': list(X.columns)}, f)
print("Saved to results/walk_forward_results.pkl")
